# Profiling da base — AprovaEdu Analytics

Diagnóstico de qualidade das 9 tabelas brutas **antes** de escrever qualquer regra de limpeza.
O objetivo aqui é *medir os problemas*, não corrigi-los — a correção fica no pipeline de ETL.

Achados que este profiling precisa confirmar antes de decidir o tratamento:

1. A nota de simulado **não tem escala única**: Redação está em 0–1000 (padrão ENEM) e as demais
   matérias em 0–100. Aplicar uma faixa 0–100 global descartaria toda a Redação.
2. A informação de *matéria* aparece em **5 tabelas** com **3 nomes de coluna** e grafias
   inconsistentes — remover acento e caixa não basta (há abreviação `Mat.`).
3. `aprovacoes_vestibular` tem mais linhas do que alunos distintos; parte é duplicidade de
   cadastro sinalizada na própria base pelo campo `chamada`.
4. A taxa de presença é muito comprimida (quase todo aluno entre ~81% e ~87%), o que limita o
   poder de qualquer comparação entre grupos.


In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

# acha a pasta data/raw subindo a partir do diretorio atual (funciona rodando de notebooks/ ou da raiz)
def achar_raw():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "data" / "raw"
        if cand.exists():
            return cand
    raise FileNotFoundError("nao achei data/raw")

RAW = achar_raw()

# BOM UTF-8 nos 9 CSVs -> ler com utf-8-sig; dtype=str e keep_default_na=False pra nao
# transformar vazio em NaN nem sujar a 1a coluna
def ler(nome):
    return pd.read_csv(RAW / f"{nome}.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)

TABELAS = ["estudantes", "professores", "ofertas_curso", "simulados", "aulas",
           "matriculas", "presencas_aulas", "resultados_simulados", "aprovacoes_vestibular"]
dfs = {t: ler(t) for t in TABELAS}
print("tabelas carregadas:", list(dfs))

tabelas carregadas: ['estudantes', 'professores', 'ofertas_curso', 'simulados', 'aulas', 'matriculas', 'presencas_aulas', 'resultados_simulados', 'aprovacoes_vestibular']


## 1. Contagem de linhas e unicidade das PKs

Confirma que li a base **completa** (não uma amostra truncada) e que cada chave primária é única.

In [2]:
PK = {
    "estudantes": "aluno_id", "professores": "professor_id", "ofertas_curso": "oferta_id",
    "simulados": "simulado_id", "aulas": "aula_id", "matriculas": "matricula_id",
    "presencas_aulas": "presenca_id", "resultados_simulados": "resultado_id",
    "aprovacoes_vestibular": "aprovacao_id",
}
esperado = {"estudantes": 812, "professores": 35, "ofertas_curso": 220, "simulados": 165,
            "aulas": 2418, "matriculas": 9452, "presencas_aulas": 74997,
            "resultados_simulados": 21510, "aprovacoes_vestibular": 354}

linhas = []
for t in TABELAS:
    n = len(dfs[t])
    dup = dfs[t][PK[t]].duplicated().sum()
    linhas.append({"tabela": t, "linhas": n, "esperado": esperado[t],
                   "bate": n == esperado[t], "pk": PK[t], "pks_duplicadas": dup})
pd.DataFrame(linhas).set_index("tabela")

,linhas,esperado,bate,pk,pks_duplicadas
tabela,,,,,
estudantes,812,812,True,aluno_id,0
professores,35,35,True,professor_id,0
ofertas_curso,220,220,True,oferta_id,0
simulados,165,165,True,simulado_id,0
aulas,2418,2418,True,aula_id,0
matriculas,9452,9452,True,matricula_id,0
presencas_aulas,74997,74997,True,presenca_id,0
resultados_simulados,21510,21510,True,resultado_id,0
aprovacoes_vestibular,354,354,True,aprovacao_id,0


## 2. Integridade referencial (FKs)

A base bruta já vem com integridade referencial intacta — nenhuma FK órfã. Confirmo isso para
não introduzir um problema que não existe e para poder confiar nos joins.

In [3]:
def fk_ok(tab_f, col_f, tab_d, col_d):
    return dfs[tab_f][col_f].isin(dfs[tab_d][col_d]).all()

checks = {
    "matriculas.oferta_id -> ofertas_curso": fk_ok("matriculas", "oferta_id", "ofertas_curso", "oferta_id"),
    "presencas.aula_id -> aulas":            fk_ok("presencas_aulas", "aula_id", "aulas", "aula_id"),
    "resultados.simulado_id -> simulados":   fk_ok("resultados_simulados", "simulado_id", "simulados", "simulado_id"),
}
for tab in ["matriculas", "presencas_aulas", "resultados_simulados", "aprovacoes_vestibular"]:
    checks[f"{tab}.aluno_id -> estudantes"] = fk_ok(tab, "aluno_id", "estudantes", "aluno_id")
pd.Series(checks, name="fk_intacta").to_frame()

,fk_intacta
matriculas.oferta_id -> ofertas_curso,True
presencas.aula_id -> aulas,True
resultados.simulado_id -> simulados,True
matriculas.aluno_id -> estudantes,True
presencas_aulas.aluno_id -> estudantes,True
resultados_simulados.aluno_id -> estudantes,True
aprovacoes_vestibular.aluno_id -> estudantes,True


## 3. Achado crítico: a nota de simulado tem escala diferente por matéria

Junto `resultados_simulados` com `simulados.materia` e olho a **mediana da nota por matéria**.
Se a escala fosse única, todas as medianas ficariam na mesma ordem de grandeza. Rodar isto
*antes* de qualquer filtro de faixa válida é o que evita o erro de descartar a Redação inteira.

In [4]:
res = dfs["resultados_simulados"].merge(dfs["simulados"][["simulado_id", "materia"]],
                                        on="simulado_id", how="left")
res["nota_num"] = pd.to_numeric(res["nota"].replace("", np.nan), errors="coerce")

# normkey: tira acento/caixa so pra agrupar grafias equivalentes na visao de profiling
def normkey(s):
    s = str(s).strip().lower()
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")

res["materia_norm"] = res["materia"].map(normkey)
esc = (res.groupby("materia_norm")["nota_num"]
          .agg(["count", "median", "mean", "min", "max"])
          .sort_values("median"))
esc

,count,median,mean,min,max
materia_norm,,,,,
mat.,116,59.50,61.593966,15.1,99.9
matematica,1765,61.20,62.099263,-1.0,1005.0
sociologia,1792,61.70,62.590792,-4.2,1005.0
fisica,1967,61.80,62.097966,-1.0,1005.0
historia,1815,61.80,62.731129,-3.5,1005.0
biologia,1843,61.90,62.713728,-1.0,1005.0
geografia,1768,61.90,62.575283,7.7,1005.0
ingles,1656,62.00,63.219324,-1.0,1005.0
quimica,1829,62.10,62.137452,-5.0,105.0


A mediana de **redacao** fica em torno de **720**, enquanto todas as outras matérias ficam perto
de **60**. É uma ordem de grandeza acima — incompatível com 0–100 e compatível com a escala ENEM
0–1000. Confirma o achado: **a faixa válida de nota tem de ser por matéria, não global**. Também
aparecem valores implausíveis dentro da própria escala (negativos, ou ~1005) — erros de
digitação a tratar como ausentes no ETL, nunca truncar.

## 4. Achado crítico: "matéria" em 5 tabelas, 3 nomes de coluna, grafias inconsistentes

Para cada uma das 5 colunas de matéria, conto as grafias brutas e quantas sobram depois de só
remover acento/caixa. A distância entre "brutas" e "após normkey" mostra que existe sujeira de
acento; e o resto (ex.: `Mat.` continuar diferente de `matematica`) mostra que **é preciso um
mapa canônico explícito** por cima da normalização.

In [5]:
COLUNAS_MATERIA = [
    ("matriculas", "materia_declarada"),
    ("aulas", "materia"),
    ("ofertas_curso", "materia"),
    ("simulados", "materia"),
    ("professores", "materia_principal"),
]
res_cols = []
for tab, col in COLUNAS_MATERIA:
    s = dfs[tab][col]
    tem_abrev = any("mat." in normkey(v) for v in s.unique())
    res_cols.append({"tabela.coluna": f"{tab}.{col}", "grafias_brutas": s.nunique(),
                     "apos_normkey": s.map(normkey).nunique(), "contem_Mat.": tem_abrev})
pd.DataFrame(res_cols).set_index("tabela.coluna")

,grafias_brutas,apos_normkey,contem_Mat.
tabela.coluna,,,
matriculas.materia_declarada,28,12,True
aulas.materia,19,11,False
ofertas_curso.materia,19,11,False
simulados.materia,16,12,True
professores.materia_principal,15,12,True


In [6]:
# a coluna mais suja da base: 28 grafias brutas
sorted(dfs["matriculas"]["materia_declarada"].unique())

['BIOLOGIA',
 'Biologia',
 'FILOSOFIA',
 'Filosofia',
 'Fisica',
 'FÍSICA',
 'Física',
 'GEOGRAFIA',
 'Geografia',
 'Historia',
 'História',
 'Ingles',
 'Inglês',
 'MATEMÁTICA',
 'Mat.',
 'Matematica',
 'Matemática',
 'PORTUGUÊS',
 'Portugues',
 'Português',
 'QUÍMICA',
 'Quimica',
 'Química',
 'REDAÇÃO',
 'Redacao',
 'Redação',
 'SOCIOLOGIA',
 'Sociologia']

`matriculas.materia_declarada` é a mais suja (28 grafias) e a mais fácil de esquecer, porque o
nome da coluna (`materia_declarada`) não segue o padrão `materia` das outras quatro. `Mat.`
aparece aqui e não é resolvido só tirando acento — precisa de mapa explícito.

## 5. Achado: duplicidade de cadastro em `aprovacoes_vestibular`

São 354 linhas para 306 alunos distintos. Parte disso é duplicidade de cadastro — e a própria
base deixou uma dica: o campo `chamada` traz o literal `"Cadastro duplicado?"` em algumas linhas.
Confirmo que cada linha marcada tem um par idêntico (mesmo aluno, ano e nota final).

In [7]:
apr = dfs["aprovacoes_vestibular"]
print("linhas:", len(apr), "| alunos distintos:", apr["aluno_id"].nunique(),
      "| alunos com >1 aprovacao:", (apr.groupby("aluno_id").size() > 1).sum())

flag = apr[apr["chamada"] == "Cadastro duplicado?"]
com_par = sum(
    len(apr[(apr["aluno_id"] == r["aluno_id"]) &
            (apr["ano_vestibular"] == r["ano_vestibular"]) &
            (apr["nota_final_vestibular"] == r["nota_final_vestibular"])]) >= 2
    for _, r in flag.iterrows()
)
print(f"linhas marcadas 'Cadastro duplicado?': {len(flag)} | com par identico confirmado: {com_par}")
print(f"aprovacoes apos remover as marcadas: {len(apr) - len(flag)}")

linhas: 354 | alunos distintos: 306 | alunos com >1 aprovacao: 46
linhas marcadas 'Cadastro duplicado?': 15 | com par identico confirmado: 15
aprovacoes apos remover as marcadas: 339


15 linhas marcadas, **todas** com par idêntico confirmado. A regra de deduplicação: remover só
essas 15 (não remover cegamente por múltiplas aprovações — 31 alunos têm aprovações genuinamente
múltiplas, em universidades/anos diferentes). Sobram **339 aprovações**.

## 6. Achado: a taxa de presença é muito comprimida

Presença efetiva = `Presente` ou `Atrasado`. Calculo a taxa por aluno sobre todas as suas aulas
e olho a distribuição. Uma variável espremida num intervalo estreito tem pouco poder de separar
aprovados de não aprovados — isso condiciona como a Q2 vai ser respondida (com teste formal e
honestidade sobre resultado fraco/nulo).

In [8]:
pres = dfs["presencas_aulas"].copy()
pres["efetiva"] = pres["status_presenca"].map(normkey).isin(["presente", "atrasado"])
taxa = pres.groupby("aluno_id")["efetiva"].mean() * 100
taxa.describe(percentiles=[.25, .5, .75]).round(1)

count    800.0
mean      84.0
std        4.1
min       70.3
25%       81.5
50%       84.2
75%       86.6
max       97.1
Name: efetiva, dtype: float64

## 7. Q1 — as três métricas que não podem ser confundidas

A taxa de aprovação não vem pronta. Há três contagens diferentes e fáceis de confundir; misturá-las
produz "taxas" absurdas. Mostro as três lado a lado para o denominador ficar auditável.

In [9]:
matr = dfs["matriculas"]
a = matr.groupby("ano")["aluno_id"].nunique()                       # matriculados distintos
b = apr.groupby("ano_vestibular").size()                             # eventos de aprovacao (bruto)
c = apr.groupby("ano_vestibular")["aluno_id"].nunique()              # aprovados distintos
comp = pd.DataFrame({"(a) matriculados_dist": a, "(b) eventos_aprov": b, "(c) aprovados_dist": c})
comp.index.name = "ano"
comp

,(a) matriculados_dist,(b) eventos_aprov,(c) aprovados_dist
ano,,,
2021,138,50,50
2022,170,53,53
2023,218,78,77
2024,263,87,79
2025,233,86,80


A taxa de aprovação correta é **(c) ÷ (a)** — aprovados distintos sobre matriculados distintos,
por ano. Nunca (b) ÷ (a), que mistura contagem de eventos com contagem de pessoas.

## 8. Formatos de data em `resultados_simulados.inicio_simulado`

A coluna mistura vários formatos (ISO, BR com barra, US com traço, com e sem hora). O parser do
ETL vai tentar os formatos em cascata, priorizando barra/BR antes de traço/US para evitar a
ambiguidade dia↔mês.

In [10]:
import re
amostra = dfs["resultados_simulados"]["inicio_simulado"]
amostra = amostra[amostra != ""].drop_duplicates().head(12)
for v in amostra:
    print(v)

28/05/2021 12:00
2021-10-10 10:00
2021-04-13 13:30
2021-06-20 13:15
2021-10-08 11:00
2021-10-18 10:00
2021-06-24
2021-10-22 08:30
2021-04-24 18:45
2021-04-10 16:15
2021/07/13 11:15
2021-10-23


## Conclusão do profiling

Os quatro achados críticos foram confirmados diretamente na base:

- **Escala de nota por matéria** (Redação ~720, demais ~60) → faixa válida por matéria no ETL.
- **5 colunas de matéria com grafias inconsistentes** → mapa canônico explícito, aplicado a todas.
- **15 duplicidades de cadastro em aprovações** confirmadas via `chamada` → remover só essas.
- **Presença comprimida** (mediana ~84%, quase todos entre 81% e 87%) → Q2 exige teste formal.

Esses achados guiam as decisões de tratamento registradas em `reports/DECISOES.md` e implementadas
no pipeline de ETL.